# XoL Tower Example

This notebook generates frequency-severity losses and applies an excess-of-loss tower using direct imports from PAL's public submodules.

In [ ]:
import numpy as np

from pal import config
from pal.contracts import XoLTower
from pal.distributions import GPD, Normal, Poisson
from pal.frequency_severity import FrequencySeverityModel

config.n_sims = 100_000

severity = GPD(shape=0.33, scale=100_000, loc=1_000_000)
frequency = Poisson(mean=2)
losses = FrequencySeverityModel(frequency, severity).generate()
losses = np.minimum(losses, 5_000_000) * 1.05
inflation = Normal(0.05, 0.02).generate()
gross_losses = losses * (1 + inflation)

In [ ]:
tower = XoLTower(
    limit=[1_000_000, 1_000_000, 1_000_000, 1_000_000, 10_000_000],
    excess=[1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000],
    aggregate_limit=[3_000_000, 2_000_000, 1_000_000, 1_000_000, 10_000_000],
    premium=[5_000, 4_000, 3_000, 2_000, 1_000],
    reinstatement_cost=[[1, 1, 1]] * 5,
)
result = tower.apply(gross_losses)
result.recoveries.aggregate().cdf_plot("Recoveries")